# Customer Churn Prediction — Exploratory Data Analysis

This notebook explores the Telco Customer Churn dataset to understand patterns and drivers of churn before building a predictive model.

In [ ]:
# Import libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import LabelEncoder

# Consistent plot style
sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Load & Overview

Load the dataset and inspect its shape, data types, first few rows, and missing values.

In [ ]:
from pathlib import Path

# Resolve data path anchored to this notebook's own file location.
# VS Code injects __vsc_ipynb_file__ with the notebook's absolute path.
_nb_file = globals().get("__vsc_ipynb_file__")
if _nb_file:
    # notebooks/analysis.ipynb -> go up one level to project root
    DATA_PATH = Path(_nb_file).resolve().parent.parent / "data" / "telco_churn.csv"
else:
    # Fallback: walk up from cwd until data/telco_churn.csv is found
    _cur = Path.cwd()
    while not (_cur / "data" / "telco_churn.csv").exists():
        if _cur.parent == _cur:
            raise FileNotFoundError("Cannot locate data/telco_churn.csv")
        _cur = _cur.parent
    DATA_PATH = _cur / "data" / "telco_churn.csv"

print(f"Data path: {DATA_PATH}")

# Load the dataset
df = pd.read_csv(DATA_PATH)

# Dataset dimensions
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns\n")

# Column data types
print("Data types:")
print(df.dtypes)
print()

In [ ]:
# First 5 rows
df.head()

In [ ]:
# Missing values per column
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct})
missing_df[missing_df["Missing Count"] > 0] if missing_df["Missing Count"].sum() > 0 else print("No missing values found.")

## 2. Class Imbalance

Understanding the distribution of the target variable `Churn` is essential. A significant imbalance can bias model training.

In [ ]:
# Value counts and percentages
churn_counts = df["Churn"].value_counts()
churn_pct = df["Churn"].value_counts(normalize=True).mul(100).round(2)
churn_summary = pd.DataFrame({"Count": churn_counts, "Percentage (%)": churn_pct})
print(churn_summary)

# Bar chart of Churn distribution
fig, ax = plt.subplots(figsize=(10, 6))
sns.countplot(data=df, x="Churn", palette=["#2ecc71", "#e74c3c"], ax=ax)
ax.set_title("Churn Distribution", fontsize=15, fontweight="bold")
ax.set_xlabel("Churn", fontsize=12)
ax.set_ylabel("Number of Customers", fontsize=12)

# Annotate bars with counts and percentages
for p in ax.patches:
    count = int(p.get_height())
    pct = count / len(df) * 100
    ax.annotate(f"{count}\n({pct:.1f}%)",
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha="center", va="bottom", fontsize=11)

plt.tight_layout()
plt.show()

## 3. Churn by Contract Type

Customers on month-to-month contracts may churn more often than those on longer-term contracts. The grouped bar chart below shows churn counts for each contract type.

In [ ]:
# Grouped bar chart: Churn by Contract type
fig, ax = plt.subplots(figsize=(10, 6))
sns.countplot(data=df, x="Contract", hue="Churn",
              palette={"No": "#2ecc71", "Yes": "#e74c3c"}, ax=ax)
ax.set_title("Churn by Contract Type", fontsize=15, fontweight="bold")
ax.set_xlabel("Contract Type", fontsize=12)
ax.set_ylabel("Number of Customers", fontsize=12)
ax.legend(title="Churn", fontsize=11)

# Annotate each bar
for p in ax.patches:
    if p.get_height() > 0:
        ax.annotate(f"{int(p.get_height())}",
                    (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()

## 4. Churn by Monthly Charges

Higher monthly charges could be associated with higher churn. The overlapping histograms below compare the distribution of `MonthlyCharges` for churned vs. retained customers.

In [ ]:
# Overlapping histogram: Monthly Charges by Churn
fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(data=df, x="MonthlyCharges", hue="Churn",
             palette={"No": "#2ecc71", "Yes": "#e74c3c"},
             bins=40, alpha=0.6, kde=True, ax=ax)
ax.set_title("Monthly Charges Distribution by Churn", fontsize=15, fontweight="bold")
ax.set_xlabel("Monthly Charges ($)", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.legend(title="Churn", fontsize=11)
plt.tight_layout()
plt.show()

## 5. Churn by Tenure

Customers with shorter tenure (newer customers) may be more likely to churn. The overlapping histograms below show tenure distributions for both groups.

In [ ]:
# Overlapping histogram: Tenure by Churn
fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(data=df, x="tenure", hue="Churn",
             palette={"No": "#2ecc71", "Yes": "#e74c3c"},
             bins=40, alpha=0.6, kde=True, ax=ax)
ax.set_title("Tenure Distribution by Churn", fontsize=15, fontweight="bold")
ax.set_xlabel("Tenure (Months)", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.legend(title="Churn", fontsize=11)
plt.tight_layout()
plt.show()

## 6. Churn by Payment Method

Certain payment methods may correlate with higher churn rates. Electronic check users, for example, might show different behaviour than customers on automatic payments.

In [ ]:
# Grouped bar chart: Churn by Payment Method
fig, ax = plt.subplots(figsize=(10, 6))
sns.countplot(data=df, x="PaymentMethod", hue="Churn",
              palette={"No": "#2ecc71", "Yes": "#e74c3c"}, ax=ax)
ax.set_title("Churn by Payment Method", fontsize=15, fontweight="bold")
ax.set_xlabel("Payment Method", fontsize=12)
ax.set_ylabel("Number of Customers", fontsize=12)
ax.legend(title="Churn", fontsize=11)
plt.xticks(rotation=15, ha="right")

# Annotate each bar
for p in ax.patches:
    if p.get_height() > 0:
        ax.annotate(f"{int(p.get_height())}",
                    (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

## 7. Correlation Heatmap

To compute correlations, all categorical columns are first label-encoded. The heatmap highlights which features are most correlated with `Churn` and with each other.

In [ ]:
# Work on a copy so the original df is unchanged
df_encoded = df.copy()

# Drop customerID if present (non-predictive)
if "customerID" in df_encoded.columns:
    df_encoded.drop(columns=["customerID"], inplace=True)

# Convert TotalCharges to numeric (may contain spaces)
df_encoded["TotalCharges"] = pd.to_numeric(df_encoded["TotalCharges"], errors="coerce")
df_encoded.dropna(inplace=True)

# Label-encode all remaining object columns (including Churn)
le = LabelEncoder()
for col in df_encoded.select_dtypes(include="object").columns:
    df_encoded[col] = le.fit_transform(df_encoded[col])

# Compute correlation matrix
corr_matrix = df_encoded.corr()

# Plot heatmap
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm",
            linewidths=0.5, annot_kws={"size": 7}, ax=ax)
ax.set_title("Feature Correlation Heatmap", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()

## 8. Key Insights

Based on the exploratory analysis above, here are the most important findings:

1. **Class Imbalance (~26% churn rate):** Only about 1 in 4 customers churns. Models trained without accounting for this will be biased toward predicting "No Churn". Using `class_weight='balanced'` or oversampling techniques is recommended.

2. **Contract Type is a strong predictor:** Month-to-month customers churn at a far higher rate than customers on one-year or two-year contracts. Locking customers into longer contracts is an effective retention lever.

3. **Higher monthly charges correlate with churn:** Churned customers tend to have higher monthly charges. This may reflect dissatisfaction with value-for-money at higher price points.

4. **Low tenure customers churn more:** The tenure histogram shows a clear spike of churners at very low tenure values (0–12 months). This suggests the first year is the most critical period for retention efforts.

5. **Electronic check payment method has the highest churn:** Customers paying by electronic check show notably higher churn than those on automatic bank transfer or credit card. This group could be targeted with incentives to switch to auto-pay.